# EXP-004 — SVM Baseline (WESAD physiological68)

**Nhóm:** Baseline G3 · **Trạng thái:** chưa xác nhận chạy lại trên máy người dùng · **Protocol:** `g3_defaults_physiological68_v1`

**Mục tiêu:** Xây dựng mốc tham chiếu cho mô hình **SVM mặc định** trên WESAD, kiểm tra khả năng tổng quát hóa sang người tham gia chưa từng xuất hiện trong train. **Giả thuyết:** Mô hình có thể học được tín hiệu phân biệt trạng thái stress, nhưng hiệu quả phải được đánh giá bằng LOSO thay vì đánh giá trên cửa sổ ngẫu nhiên.

Notebook này sử dụng **runner G3 chính thức** của dự án; không sao chép hay viết lại thuật toán đánh giá. Các chỉ số chỉ được trình bày từ đầu ra đã lưu và xác thực; không có kết quả bịa trước.

## 1. Môi trường, đường dẫn, điều kiện chạy

- Tạo môi trường theo `requirements.lock.txt` của nhóm và cài thêm `matplotlib`, `ipykernel` nếu cần.
- Trong VS Code/Jupyter, chọn đúng kernel Python đã cài phụ thuộc; các lệnh `%pip` cài vào **kernel hiện tại**.
- Đặt dữ liệu G2 tại đường dẫn chỉ ra trong `config.yaml`; không đưa bảng feature lớn vào Git.
- Chạy notebook từ vị trí nào cũng được. Mặc định **không huấn luyện** cho đến khi chủ động bật cờ.
- Nếu lỗi, xem `STDERR` in rõ bên dưới. Không tự thay checksum, protocol hoặc chỉnh sửa kết quả để vượt qua kiểm tra.

In [ ]:
from pathlib import Path
import sys
import json
import subprocess
import importlib.util
import hashlib

EXP_ID = "EXP-004"
MODEL_NAME = "svm"
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents)
             if (p / "scripts/run_g3_baselines.py").is_file() and (p / "configs/defaults.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục gốc wesad-stress-detection; mở đúng dự án trên VS Code.")
EXP_DIR = ROOT / "experiments/01_baselines/EXP-004_SVM"
CONFIG_PATH = EXP_DIR / "config.yaml"
assert CONFIG_PATH.is_file(), f"Thiếu {CONFIG_PATH}"
print("Python kernel:", sys.executable)
print("Project root:", ROOT)
print("Config:", CONFIG_PATH)

required = {"yaml": "PyYAML", "imblearn": "imbalanced-learn", "sklearn": "scikit-learn", "pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Thiếu gói:", missing)
    print("Chạy trong cell riêng: %pip install -r", str(ROOT / "requirements.lock.txt"))
    print("Sau đó: %pip install matplotlib ipykernel; restart kernel")
    raise ModuleNotFoundError("Thiếu phụ thuộc notebook: " + ", ".join(missing))


In [ ]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
assert CONFIG["models"] == [MODEL_NAME] and CONFIG["tasks"] == ["multiclass", "binary"]
assert CONFIG["protocol"] == "g3_defaults_physiological68_v1"
assert not CONFIG["nested"] and not CONFIG["balance"]
FEATURE_PATH = ROOT / CONFIG["features"]
SCHEMA_PATH = ROOT / CONFIG["schema"]
MAIN_OUTPUT = ROOT / CONFIG["output_dir"]
QUICK_OUTPUT = ROOT / f"outputs/notebook_research/{EXP_ID}/r01_quick_S2"
print("Dữ liệu:", FEATURE_PATH, " | Tồn tại:", FEATURE_PATH.is_file())
print("Schema tồn tại:", SCHEMA_PATH.is_file())
print("Main output:", MAIN_OUTPUT)


## 2. Kiểm tra đầu vào và phân bố nhãn

G2 đã chốt 68 đặc trưng sinh lý, 15 subject, cửa sổ 60 giây bước 5 giây. Kiểm tra mã SHA-256 đã khóa trong cấu hình. Nếu thiếu dữ liệu, notebook vẫn xem được phần phương pháp; **không** tiến hành chạy huấn luyện. Mỗi subject được giữ nguyên cho LOSO (không trộn cửa sổ chồng lấn giữa train/test).

In [ ]:
if FEATURE_PATH.is_file() and SCHEMA_PATH.is_file():
    sha = hashlib.sha256(FEATURE_PATH.read_bytes()).hexdigest()
    print("Feature SHA256:", sha)
    assert sha == CONFIG["feature_sha256"], "Sai checksum bộ G2; kiểm tra đúng phiên bản thay vì sửa SHA trong config."
    sys.path.insert(0, str(ROOT / "src"))
    from wesad.data import load_feature_table
    frame, feature_columns = load_feature_table(FEATURE_PATH, SCHEMA_PATH)
    assert len(feature_columns) == 68, f"Phải có 68 đặc trưng, hiện có {len(feature_columns)}"
    assert set(frame.subject_id) == set(CONFIG["expected_subjects"]), "Không đủ 15 subject hoặc lẫn subject"
    print("Số cửa sổ:", len(frame), "| Số subject:", frame.subject_id.nunique(), "| Features:", len(feature_columns))
    display(pd.crosstab(frame.subject_id, frame.label_original, margins=True))
else:
    print("CHƯA CÓ DỮ LIỆU: tải gói G2 physiological68 của nhóm theo README dự án; không thể chạy LOSO.")


## 3. Pipeline và tham số baseline

- Mô hình: `SVC(C=1.0, kernel="rbf", gamma="scale")` (giá trị mặc định `scikit-learn`).
- Xử lý đầu vào: `SimpleImputer(strategy="median", keep_empty_features=True)` → `StandardScaler` → mô hình.
- Imputer và scaler chỉ được `fit` trên **14 subject train** của từng outer fold.
- **Không** tuning, không class balancing, không feature selection.
- Hai bài toán: multiclass `[1,2,3]` và binary `[0,1]` với stress = 1.
- Đánh giá: LOSO 15 subject; Macro-F1 theo nhãn cố định, Balanced Accuracy và confusion matrix.

In [ ]:
sys.path.insert(0, str(ROOT / "src"))
from wesad.models import build_pipeline
pipeline = build_pipeline(MODEL_NAME, CONFIG)
print(pipeline)
print("Model parameters:", pipeline.named_steps["model"].get_params())


## 4. R01 — Chạy thử S2 (quick, không phải kết quả chính thức)

Chạy chính xác 2 fold (một task multiclass, một binary) cho người S2, dùng 14 người còn lại huấn luyện. `--quick-subject S2` và output chứa chữ `quick` để runner G3 chấp nhận đúng phạm vi. Nếu cần chạy lại sau khi sửa code/môi trường, sử dụng một **output_dir mới** để không ghi đè bằng chứng cũ.

In [ ]:
RUN_QUICK = True  # Đổi thành True khi đã kiểm tra dữ liệu và môi trường
RUN_MAIN = True   # Chỉ bật sau khi R01 đã chạy và verify thành công

def run_and_show(command):
    print("COMMAND:", " ".join(map(str, command)))
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                            encoding="utf-8", errors="replace")
    print("RETURN CODE:", result.returncode)
    if result.stdout.strip():
        print("--- STDOUT ---\n" + result.stdout[-15000:])
    if result.stderr.strip():
        print("--- STDERR (nguyên nhân lỗi) ---\n" + result.stderr[-25000:])
    if result.returncode != 0:
        raise RuntimeError(f"Runner thất bại (code {result.returncode}); đọc STDERR phía trên.")
    return result

if RUN_QUICK:
    assert FEATURE_PATH.is_file() and SCHEMA_PATH.is_file(), "Thiếu dữ liệu/schema G2"
    cmd = [sys.executable, "scripts/run_g3_baselines.py", "--config", str(CONFIG_PATH),
           "--quick-subject", "S2", "--output", str(QUICK_OUTPUT), "--resume"]
    run_and_show(cmd)
else:
    print("R01 chưa chạy: đặt RUN_QUICK=True khi sẵn sàng.")


In [ ]:
quick_verification = QUICK_OUTPUT / "verification.json"
if quick_verification.is_file():
    qv = json.loads(quick_verification.read_text(encoding="utf-8"))
    print("R01 verify:", qv.get("status"), "| scope:", qv.get("scope"),
          "| folds:", qv.get("n_folds"), "| predictions:", qv.get("n_predictions"))
    if qv.get("status") != "passed" or qv.get("scope") != "quick":
        print("CẢNH BÁO: Quick chưa đạt; không nên chạy main.")
    quick_metrics = QUICK_OUTPUT / "verified_metrics_by_subject.csv"
    if quick_metrics.is_file():
        display(pd.read_csv(quick_metrics))
else:
    print("Chưa có verification.json của R01; không được báo cáo số liệu R01 như đã chạy.")


## 5. R02 — Chạy chính thức đầy đủ 15 subject

Bật `RUN_MAIN=True` ở cell R01 rồi chạy cell dưới. **R02** tạo 30 folds (15 subject × 2 tasks), không dùng kết quả quick. Runner G3 kiểm tra hash, source, config, môi trường, lưu prediction OOF và verification. `--resume` chỉ tái sử dụng fold có kiểm tra dấu vết hợp lệ.

In [ ]:
if RUN_MAIN:
    assert FEATURE_PATH.is_file() and SCHEMA_PATH.is_file(), "Thiếu dữ liệu/schema G2"
    if not quick_verification.is_file():
        raise RuntimeError("Chạy R01 và kiểm tra verification.json trước R02.")
    qv = json.loads(quick_verification.read_text(encoding="utf-8"))
    assert qv.get("status") == "passed" and qv.get("scope") == "quick", "R01 chưa được xác thực"
    run_and_show([sys.executable, "scripts/run_g3_baselines.py", "--config", str(CONFIG_PATH), "--resume"])
else:
    print("R02 chưa chạy: đặt RUN_MAIN=True ở cell R01 để chạy chính thức.")


## 6. Kết quả R02 theo người và tổng hợp

Chỉ đọc từ `verified_metrics_by_subject.csv` khi `verification.json` là `passed` và `scope=main`. Thống kê mean/std theo 15 subject (không gộp các cửa sổ như các mẫu độc lập).

In [ ]:
vfile = MAIN_OUTPUT / "verification.json"
valid_main = False
if vfile.is_file():
    verified = json.loads(vfile.read_text(encoding="utf-8"))
    valid_main = (verified.get("status") == "passed" and verified.get("scope") == "main"
                  and verified.get("n_folds") == 30 and verified.get("n_subjects") == 15)
    print("Verification:", {k:verified.get(k) for k in ("status","scope","n_folds","n_subjects","n_predictions")})
else:
    print("Chưa có xác minh R02. Không có kết quả chạy mới để báo cáo.")

if valid_main:
    results = pd.read_csv(MAIN_OUTPUT / "verified_metrics_by_subject.csv")
    assert set(results["model"]) == {MODEL_NAME} and len(results) == 30
    display(results.sort_values(["task","subject_id"]))
    summary = results.groupby("task")[["macro_f1","balanced_accuracy"]].agg(["mean","std"])
    display(summary.round(4))
else:
    print("Chưa đủ 30 LOSO folds đã xác minh; không tính kết quả mới.")


## 7. Biểu đồ theo người và confusion matrix (OOF)

Hai đồ thị chỉ được vẽ từ dữ liệu **đã xác minh** của chính EXP này. Bảng confusion matrix được cộng từ các dự đoán held-out (không phải trên train).

In [ ]:
if valid_main:
    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
    preds = pd.read_csv(MAIN_OUTPUT / "predictions.csv")
    assert set(preds["model"]) == {MODEL_NAME}
    for task, labels in [("binary", [0,1]), ("multiclass", [1,2,3])]:
        sub = results.loc[results.task == task].sort_values("subject_id")
        ax = sub.plot(x="subject_id", y=["macro_f1","balanced_accuracy"], kind="bar", figsize=(11,4), rot=45,
                      title=f"{EXP_ID} {MODEL_NAME}: {task} — LOSO by subject")
        ax.set_ylim(0,1.05); ax.set_ylabel("Score"); plt.tight_layout(); plt.show()
        oof = preds.loc[preds.task == task]
        cm = confusion_matrix(oof.y_true, oof.y_pred, labels=labels)
        ConfusionMatrixDisplay(cm, display_labels=labels).plot(values_format="d", cmap="Blues")
        plt.title(f"{EXP_ID}: {task} (out-of-fold)"); plt.tight_layout(); plt.show()
else:
    print("Bỏ qua hình R02: chưa có dữ liệu OOF đã xác minh.")


## 8. Đối chiếu Dummy / LR và các baseline G3

Nếu có `reports/g3_group/metrics_by_subject.csv` thì đọc riêng như **nguồn lịch sử của nhóm**, không nhầm với lần chạy R02 mới. Chỉ so sánh trên cùng task, giao các subject; so sánh có ý nghĩa khi bộ feature và protocol nhất quán. Chênh lệch không phải bằng chứng thống kê nếu chưa kiểm định.

In [ ]:
group_file = ROOT / "reports/g3_group/metrics_by_subject.csv"
if group_file.is_file():
    historical = pd.read_csv(group_file)
    if {"task","model","subject_id","macro_f1","balanced_accuracy"}.issubset(historical.columns):
        historical = historical.loc[historical.model.isin(["dummy", "lr", MODEL_NAME])]
        display(historical.groupby(["task","model"])[["macro_f1","balanced_accuracy"]].mean().round(4))
        if valid_main:
            baseline = historical.loc[historical.model.isin(["dummy", "lr"])]
            joined = results.merge(baseline, on=["task","subject_id"], suffixes=("_ours","_baseline"))
            joined["delta_macro_f1"] = joined["macro_f1_ours"] - joined["macro_f1_baseline"]
            display(joined.groupby(["task","model_baseline"])["delta_macro_f1"].agg(["count","mean","std"]).round(4))
    else:
        print("Bảng G3 không có đủ cột yêu cầu; không tính đối chiếu.")
else:
    print("Không tìm thấy báo cáo lịch sử G3; có thể bổ sung sau khi đồng bộ repo.")


## 9. Xuất CSV và ảnh PNG cho báo cáo

Chạy cell dưới sau R02. **Chỉ xuất khi** `verification.json` báo `passed`, scope `main`, đủ 15 subjects và 30 folds. Lưu `results.csv`, `summary.csv`, `figures/` và metadata truy nguồn ngay trong module; dữ liệu dự đoán chi tiết vẫn được runner lưu tại `outputs/`. Biểu đồ đối chiếu Dummy/LR chỉ sinh nếu tìm được báo cáo G3 lịch sử, với chú thích nguồn rõ ràng. Không đưa số liệu giả vào Git.


In [ ]:
# Xuất bằng chứng nghiên cứu chỉ sau khi 30 LOSO folds được xác minh.
# Đầu ra dưới experiments/01_baselines/EXP-00X_.../ để dễ review trên GitHub.
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import shutil

if not valid_main:
    print("Chưa có R02 verification PASSED (30 folds / 15 subjects): KHÔNG xuất CSV/PNG.")
else:
    FIG_DIR = EXP_DIR / "figures"
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    expected_tasks = {"binary", "multiclass"}
    assert set(results.task) == expected_tasks and set(results.model) == {MODEL_NAME}
    assert results.groupby("task").subject_id.nunique().to_dict() == {"binary":15,"multiclass":15}
    oof = pd.read_csv(MAIN_OUTPUT / "predictions.csv", dtype={"subject_id":str, "window_id":str})
    assert set(oof.task) == expected_tasks and set(oof.model) == {MODEL_NAME}
    assert not oof.duplicated(["task","model","subject_id","window_id"]).any()
    assert oof.groupby("task").subject_id.nunique().to_dict() == {"binary":15,"multiclass":15}

    # CSV gắn với chính dữ liệu dự đoán đã verified của runner G3.
    results.sort_values(["task","subject_id"]).to_csv(EXP_DIR / "results.csv", index=False)
    summary_csv = (results.groupby(["task","model"], as_index=False)
                   .agg(n_subjects=("subject_id","nunique"),
                        macro_f1_mean=("macro_f1","mean"),
                        macro_f1_std=("macro_f1","std"),
                        balanced_accuracy_mean=("balanced_accuracy","mean"),
                        balanced_accuracy_std=("balanced_accuracy","std")))
    summary_csv.to_csv(EXP_DIR / "summary.csv", index=False)
    # OOF có thể lớn: đặt trong outputs của runner, không nhân đôi trong Git.

    for task, labels, class_names in [("binary",[0,1],["Non-stress","Stress"]),
                                      ("multiclass",[1,2,3],["Baseline","Stress","Amusement"])]:
        sub = results.loc[results.task == task].sort_values("subject_id")
        x = np.arange(len(sub))
        fig, ax = plt.subplots(figsize=(12,4.5))
        ax.bar(x-0.2, sub["macro_f1"].to_numpy(), width=0.4, label="Macro-F1")
        ax.bar(x+0.2, sub["balanced_accuracy"].to_numpy(), width=0.4, label="Balanced accuracy")
        ax.set_xticks(x, sub["subject_id"].astype(str), rotation=45)
        ax.set_ylim(0,1.05); ax.set_ylabel("Score")
        ax.set_title(f"{EXP_ID}: {task} - 15 held-out subjects")
        ax.legend(); fig.tight_layout()
        fig.savefig(FIG_DIR / f"metrics_by_subject_{task}.png", dpi=160, bbox_inches="tight")
        plt.show(); plt.close(fig)

        p = oof.loc[oof.task == task]
        cm = confusion_matrix(p.y_true, p.y_pred, labels=labels)
        fig, ax = plt.subplots(figsize=(6,5))
        ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, values_format="d", cmap="Blues", colorbar=False)
        ax.set_title(f"{EXP_ID}: {task} (LOSO OOF)")
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"confusion_matrix_{task}.png", dpi=160, bbox_inches="tight")
        plt.show(); plt.close(fig)

    # Chỉ vẽ so sánh khi bảng G3 lịch sử thực sự có đủ thông tin.
    history_path = ROOT / "reports/g3_group/metrics_by_subject.csv"
    comparison_image = FIG_DIR / "comparison_baselines.png"
    if history_path.is_file():
        hist = pd.read_csv(history_path, dtype={"subject_id":str})
        needed = {"task","model","subject_id","macro_f1","balanced_accuracy"}
        if needed.issubset(hist.columns):
            baseline = hist.loc[hist.model.isin(["dummy","lr"]) & hist.task.isin(expected_tasks)].copy()
            if not baseline.empty:
                # Chỉ đối chiếu trên giao các subject cho từng task / baseline.
                matched = results.merge(baseline, on=["task","subject_id"], suffixes=("_ours","_baseline"))
                if not matched.empty:
                    matched["delta_macro_f1"] = matched.macro_f1_ours - matched.macro_f1_baseline
                    matched["delta_balanced_accuracy"] = (matched.balanced_accuracy_ours - matched.balanced_accuracy_baseline)
                    matched.to_csv(EXP_DIR / "comparison_baselines.csv", index=False)
                    grouped = (matched.groupby(["task","model_baseline"],as_index=False)
                               .agg(macro_f1_delta_mean=("delta_macro_f1","mean"),
                                    n_subjects=("subject_id","nunique")))
                    fig, ax = plt.subplots(figsize=(8,4.5))
                    names = grouped["task"] + " / " + grouped["model_baseline"]
                    ax.bar(names, grouped["macro_f1_delta_mean"])
                    ax.axhline(0, color="gray", linewidth=1)
                    ax.set_ylabel("Mean paired delta Macro-F1")
                    ax.set_title(f"{EXP_ID} vs historical G3 baselines (matched subjects)")
                    ax.tick_params(axis="x", labelrotation=25)
                    fig.tight_layout()
                    fig.savefig(comparison_image, dpi=160, bbox_inches="tight")
                    plt.show(); plt.close(fig)
                else:
                    print("Không có cùng subject/task để đối chiếu Dummy/LR.")
            else:
                print("Bảng G3 không chứa Dummy/LR; không xuất ảnh so sánh.")
        else:
            print("Thiếu cột trong báo cáo lịch sử G3; không xuất ảnh so sánh.")
    else:
        print("Chưa có reports/g3_group/metrics_by_subject.csv; bỏ qua ảnh so sánh.")

    # Giữ metadata gốc, phục vụ kiểm chứng và truy nguồn kết quả.
    provenance = {
       "experiment": EXP_ID, "model":MODEL_NAME,
       "source_verified_directory":str(MAIN_OUTPUT.relative_to(ROOT)),
       "verification_status":verified["status"], "scope":verified["scope"],
       "n_folds":verified["n_folds"], "n_subjects":verified["n_subjects"],
       "feature_sha256":CONFIG["feature_sha256"],
       "note":"Files regenerated from verified R02; no fabricated test results."
    }
    (EXP_DIR / "result_provenance.json").write_text(json.dumps(provenance, indent=2, ensure_ascii=False)+"\n", encoding="utf-8")
    print("Đã xuất CSV và PNG vào:", EXP_DIR)
    print("Giữ predictions.csv và từng fold chi tiết tại:", MAIN_OUTPUT)


## 10. Nhận xét và kết luận (chỉ điền sau khi có kết quả)

- **Cấu hình:** EXP-004 dùng SVM mặc định, không tuning, không cân bằng lớp, dùng cùng G2 sinh lý 68 với nhóm.
- **R01:** Ghi Macro-F1/Balanced Accuracy của S2 theo hai bài toán và các cảnh báo từ verification.
- **R02:** Ghi trung bình ± độ lệch chuẩn của 15 người cho binary và multiclass; nhận xét chênh lệch giữa các subject.
- **Đối chứng:** So sánh cặp theo subject với Dummy/LR trên cùng bộ dữ liệu/protocol; không kết luận ưu việt khi nguồn khác nhau.
- **Giới hạn:** WESAD có 15 subject, các cửa sổ chồng lấn cao và nhiều biến thiên cá nhân; chỉ số train hoặc random window split không thay thế LOSO.

**Trạng thái ban đầu của notebook:** chưa có kết quả thực nghiệm R01/R02; số liệu xuất hiện sau khi chạy và xác minh.